# 03. Horizontal calibrations pipeline

### 03. Horizontal calibration pipeline

This notebook calibrates horizontal waggle dance annotations.

It uses the compass calibration summary produced by Notebook 02 and applies it to horizontal dance annotation files.

This notebook has two calibration branches:

1. Direct-compass calibration for dances with a valid assigned compass calibration.
2. Landmark-transfer calibration for dances where the camera moved without a direct compass calibration.

The final output is one combined horizontal calibrated waggle-run file.

### Imports

In [2]:
from pathlib import Path
import ast
import re
import numpy as np
import pandas as pd

## Horizontal DIRECT-COMPASS calibration

This section applies the compass calibration summary to horizontal dances that have a direct assigned calibration video.

Rows marked `include = no` in the mapping file are excluded here. These excluded rows include dances that require landmark-transfer calibration and are processed separately later.

In [ ]:
HORIZONTAL_CALIBRATION_SUMMARY_FILE = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\north_calibrations\horizontal_north_calibrations\horizontal_direct_calibrations\calibration_summary_ROBUST.csv"
)

In [ ]:
# ---------------------------------------------------------------
# Create horizontal_direct_df from the filled direct-calibration mapping
# ---------------------------------------------------------------
#
# This cell uses the manually checked mapping file:
# horizontal_direct_calibration_MAPPING_FILLED.csv
#
# It extracts waggle runs from all horizontal dances marked include == yes.
# Rows marked include == no are skipped here and processed later in the
# landmark-transfer section.
#
# Output from this cell:
# horizontal_direct_df
#
# This is then used by the next cell, which merges in
# calibration_summary_ROBUST.csv and calculates waggle_true_bearing_deg.


# ---------------------------------------------------------------
# Paths
# ---------------------------------------------------------------

# ---------------------------------------------------------------
# Paths
# ---------------------------------------------------------------

CORRECTED_ROOT = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files"
)

# This folder contains the actual horizontal waggle dance annotation CSVs,
# sorted into date subfolders.
HORIZONTAL_DANCE_ROOT = CORRECTED_ROOT / "all_horizontal_dances"

HORIZONTAL_DIRECT_CALIBRATION_ROOT = (
    CORRECTED_ROOT
    / "north_calibrations"
    / "horizontal_north_calibrations"
    / "horizontal_direct_calibrations"
)

HORIZONTAL_MAPPING_FILE = (
    HORIZONTAL_DIRECT_CALIBRATION_ROOT
    / "horizontal_direct_calibration_MAPPING_FILLED.csv"
)

OUTPUT_DIRECT_FILE = (
    HORIZONTAL_DIRECT_CALIBRATION_ROOT
    / "horizontal_direct_dances_with_calibrations.csv"
)

print("Horizontal dance folder exists:", HORIZONTAL_DANCE_ROOT.exists())
print("Horizontal dance folder:", HORIZONTAL_DANCE_ROOT)

print("\nMapping file exists:", HORIZONTAL_MAPPING_FILE.exists())
print("Mapping file:", HORIZONTAL_MAPPING_FILE)

print("\nOutput direct file will be saved to:")
print(OUTPUT_DIRECT_FILE)


# ---------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------

def parse_annotation_list(value, column_name=""):
    if pd.isna(value) or str(value).strip() == "":
        return []

    value = str(value).strip()

    value = re.sub(
        r"np\.float(?:16|32|64)?\(([^)]+)\)",
        r"\1",
        value
    )

    value = re.sub(r"\bnan\b", "None", value, flags=re.IGNORECASE)

    # Repair common missing opening bracket issue:
    # "(1, 2), (3, 4)]" -> "[(1, 2), (3, 4)]"
    if value.startswith("(") and value.endswith("]"):
        value = "[" + value

    try:
        return list(ast.literal_eval(value))
    except (ValueError, SyntaxError) as error:
        raise ValueError(
            f"Could not parse column '{column_name}': {value}"
        ) from error


def vector_to_video_angle_deg(u, v):
    """
    Video convention:
    0°   = top/up
    90°  = right
    180° = bottom/down
    270° = left
    """
    return np.degrees(np.arctan2(u, -v)) % 360


def clean_calibration_id(filename):
    name = Path(str(filename)).name.strip()

    if name.endswith(".csv"):
        name = name[:-4]

    if name.endswith("_waggle_annotations"):
        name = name.replace("_waggle_annotations", "")

    return name


def strip_annotation_suffix(filename):
    name = Path(str(filename)).name.strip()

    if name.endswith(".csv"):
        name = name[:-4]

    if name.endswith("_waggle_annotations"):
        name = name.replace("_waggle_annotations", "")

    return name


def resolve_horizontal_dance_file(row):
    """
    Resolve the horizontal dance annotation file from the mapping row.

    Priority:
    1. Use relative_path if available.
    2. Use dance_file filename inside HORIZONTAL_DANCE_ROOT.
    3. Recursive search by filename.
    """

    # 1. Use relative_path if available
    if "relative_path" in row.index and pd.notna(row["relative_path"]):
        relative_path = str(row["relative_path"]).strip()

        if relative_path != "" and relative_path.lower() != "nan":
            candidate = HORIZONTAL_DANCE_ROOT / Path(relative_path)

            if candidate.exists():
                return candidate

    # 2. Use dance_file
    dance_file_name = str(row["dance_file"]).strip()

    candidate = HORIZONTAL_DANCE_ROOT / dance_file_name

    if candidate.exists():
        return candidate

    # 3. Recursive search
    matches = sorted(HORIZONTAL_DANCE_ROOT.rglob(dance_file_name))

    if len(matches) == 1:
        return matches[0]

    if len(matches) > 1:
        raise ValueError(
            f"Multiple matches found for {dance_file_name}. "
            "Use relative_path in the mapping file to disambiguate.\n\n"
            + "\n".join(str(match) for match in matches)
        )

    raise FileNotFoundError(
        f"Could not find horizontal dance file: {dance_file_name}"
    )


def extract_horizontal_waggle_runs(csv_file, mapping_row):
    """
    Extract one row per valid waggle run from a horizontal dance annotation CSV.
    Metadata comes from the filled mapping file.
    """

    csv_file = Path(csv_file)
    df = pd.read_csv(csv_file)

    records = []

    date = str(mapping_row.get("date", "")).strip()
    bee_id = str(mapping_row.get("bee_id", "")).strip()
    dance_id = strip_annotation_suffix(mapping_row.get("dance_file", csv_file.name))
    source_batch = f"horizontal_direct_{date}"
    dance_key = f"{source_batch}__{bee_id}__{dance_id}"

    waggle_run_number = 0

    for session_index, row in df.iterrows():

        start_positions = parse_annotation_list(
            row["waggle_start_positions"],
            "waggle_start_positions"
        )

        start_frames = parse_annotation_list(
            row["waggle_start_frames"],
            "waggle_start_frames"
        )

        directions = parse_annotation_list(
            row["waggle_directions"],
            "waggle_directions"
        )

        if not (len(start_positions) == len(start_frames) == len(directions)):
            raise ValueError(
                f"{csv_file.name}: start positions, frames, and directions have different lengths."
            )

        for (start_x, start_y), start_frame, direction in zip(
            start_positions,
            start_frames,
            directions
        ):
            u, v = direction

            if u is None or v is None:
                continue

            u = float(u)
            v = float(v)

            if not np.isfinite(u) or not np.isfinite(v):
                continue

            if np.hypot(u, v) < 1e-8:
                continue

            waggle_run_number += 1

            video_angle_deg = vector_to_video_angle_deg(u, v)

            records.append({
                "source_batch": source_batch,
                "dance_key": dance_key,
                "dance_id": dance_id,
                "bee_id": bee_id,
                "date": date,
                "time": mapping_row.get("time", pd.NA),
                "condition": mapping_row.get("condition", "horizontal"),
                "notes": mapping_row.get("notes", ""),
                "recording_type": "horizontal",
                "orientation_method": "direct_compass",

                "waggle_run": waggle_run_number,
                "csv_file": csv_file.name,
                "video_name": row["video_name"],

                "direction_u": u,
                "direction_v": v,
                "video_angle_deg": video_angle_deg,

                "start_x": start_x,
                "start_y": start_y,
                "start_xy": (start_x, start_y),
                "start_frame": start_frame,
                "annotation_session": session_index + 1,

                "calibration_file": mapping_row.get("calibration_file", pd.NA),
                "calibration_id": clean_calibration_id(
                    mapping_row.get("calibration_file", pd.NA)
                )
            })

    return pd.DataFrame(records)


# ---------------------------------------------------------------
# Load and filter filled mapping file
# ---------------------------------------------------------------

mapping_df = pd.read_csv(
    HORIZONTAL_MAPPING_FILE,
    dtype=str,
    encoding="utf-8-sig"
)

mapping_df.columns = mapping_df.columns.str.strip()

print("\nRows in filled mapping file:", len(mapping_df))

if "include" in mapping_df.columns:
    include_clean = (
        mapping_df["include"]
        .fillna("yes")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    excluded_df = mapping_df.loc[
        ~include_clean.isin(["yes", "y", "true", "1"])
    ].copy()

    mapping_df = mapping_df.loc[
        include_clean.isin(["yes", "y", "true", "1"])
    ].copy()

    print("Rows excluded because include != yes:", len(excluded_df))

    if len(excluded_df) > 0:
        display(
            excluded_df[
                [
                    "date",
                    "bee_id",
                    "dance_file",
                    "include",
                    "orientation_method",
                    "notes"
                ]
            ]
        )

print("Rows kept for direct-compass calibration:", len(mapping_df))


# ---------------------------------------------------------------
# Extract direct horizontal waggle runs
# ---------------------------------------------------------------

all_direct_waggle_dfs = []
failed_rows = []

for idx, row in mapping_df.iterrows():

    try:
        dance_file = resolve_horizontal_dance_file(row)

        waggle_df = extract_horizontal_waggle_runs(
            dance_file,
            row
        )

        all_direct_waggle_dfs.append(waggle_df)

        print(
            f"Processed row {idx + 1}: "
            f"{row['dance_file']} "
            f"({len(waggle_df)} waggle runs)"
        )

    except Exception as error:
        print(f"\nCould not process row {idx + 1}: {row.get('dance_file', '')}")
        print(error)

        failed_rows.append({
            "row_number": idx + 1,
            "error": str(error),
            **row.to_dict()
        })


if len(all_direct_waggle_dfs) == 0:
    raise ValueError("No direct horizontal dances were processed successfully.")

horizontal_direct_df = pd.concat(
    all_direct_waggle_dfs,
    ignore_index=True
)

print("\nCreated horizontal_direct_df")
print("Rows:", len(horizontal_direct_df))
print("Dances:", horizontal_direct_df["dance_key"].nunique())
print("Bees:", horizontal_direct_df["bee_id"].nunique())

if failed_rows:
    failed_direct_df = pd.DataFrame(failed_rows)

    FAILED_DIRECT_ROWS_FILE = OUTPUT_DIRECT_FILE.with_name(
        "horizontal_direct_failed_rows.csv"
    )

    failed_direct_df.to_csv(
        FAILED_DIRECT_ROWS_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print("\nSome direct rows failed. Saved failed-row report to:")
    print(FAILED_DIRECT_ROWS_FILE)

    display(failed_direct_df)
else:
    print("\nAll direct horizontal mapping rows processed successfully.")

In [ ]:
calibration_summary_df = pd.read_csv(
    HORIZONTAL_CALIBRATION_SUMMARY_FILE,
    dtype=str,
    encoding="utf-8-sig"
)

calibration_summary_df.columns = calibration_summary_df.columns.str.strip()

calibration_summary_df["calibration_id"] = calibration_summary_df["calibration_file"].apply(
    clean_calibration_id
)

horizontal_direct_df["calibration_id"] = horizontal_direct_df["calibration_id"].apply(
    clean_calibration_id
)

horizontal_direct_with_cal_df = horizontal_direct_df.merge(
    calibration_summary_df,
    on="calibration_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_from_calibration_summary")
)

if "calibration_file_from_calibration_summary" in horizontal_direct_with_cal_df.columns:
    horizontal_direct_with_cal_df = horizontal_direct_with_cal_df.rename(
        columns={
            "calibration_file": "assigned_calibration_file",
            "calibration_file_from_calibration_summary": "calibration_file"
        }
    )

horizontal_direct_with_cal_df["video_angle_deg"] = pd.to_numeric(
    horizontal_direct_with_cal_df["video_angle_deg"],
    errors="coerce"
)

horizontal_direct_with_cal_df["true_north_video_deg"] = pd.to_numeric(
    horizontal_direct_with_cal_df["true_north_video_deg"],
    errors="coerce"
)

horizontal_direct_with_cal_df["waggle_true_bearing_deg"] = (
    horizontal_direct_with_cal_df["video_angle_deg"]
    - horizontal_direct_with_cal_df["true_north_video_deg"]
) % 360

horizontal_direct_with_cal_df.loc[
    horizontal_direct_with_cal_df["true_north_video_deg"].isna(),
    "waggle_true_bearing_deg"
] = np.nan

missing_cal = horizontal_direct_with_cal_df.loc[
    horizontal_direct_with_cal_df["calibration_id"].notna()
    & horizontal_direct_with_cal_df["true_north_video_deg"].isna(),
    ["assigned_calibration_file", "calibration_id"]
].drop_duplicates()

if len(missing_cal) == 0:
    print("All assigned calibrations matched calibration_summary_ROBUST.csv.")
else:
    print("These calibration assignments still did not match calibration_summary_ROBUST.csv:")
    display(missing_cal)

horizontal_direct_with_cal_df.to_csv(
    OUTPUT_DIRECT_FILE,
    index=False
)

print(f"\nSaved direct horizontal calibrated file to:\n{OUTPUT_DIRECT_FILE}")

## Horizontal LANDMARK-TRANSFER calibrations
For horizontal waggle videos that do not have a suitable compass calibration video using the same annotated landmark in calibration and waggle dance video

In [3]:
# Set path to a folder containing waggle dance videos that need a landmark-transfer calculation
NEED_CALC_ROOT = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\north_calibrations\horizontal_north_calibrations\horizontal_landmark_transfer_calibrations"
)

COMPASS_ROOT = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\north_calibrations\horizontal_north_calibrations\horizontal_compass_calibrations"
)

LANDMARK_MAPPING_FILE = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\north_calibrations\horizontal_north_calibrations\horizontal_landmark_transfer_calibrations\horizontal_landmark_transfer_calibration_MAPPING_FILLED.csv"
)

OUTPUT_WAGGLE_SUMMARY = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\horizontal_landmark_calibrated_waggle_runs.csv"
)

OUTPUT_ORIENTATION_SUMMARY = Path(
    r"C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\horizontal_landmark_calibration_summary.csv"
)

# West declination should be negative
DECLINATION_DEG = -21.27

In [4]:
def parse_annotation_list(value, column_name=""):
    if pd.isna(value) or str(value).strip() == "":
        return []

    value = str(value)

    value = re.sub(
        r"np\.float(?:16|32|64)?\(([^)]+)\)",
        r"\1",
        value
    )

    value = re.sub(r"\bnan\b", "None", value, flags=re.IGNORECASE)

    try:
        return list(ast.literal_eval(value))
    except (ValueError, SyntaxError) as error:
        raise ValueError(
            f"Could not parse column '{column_name}': {value}"
        ) from error


def vector_to_video_angle_deg(u, v):
    """
    Your video convention:
    0°   = top/up
    90°  = right
    180° = bottom/down
    270° = left
    """
    return np.degrees(np.arctan2(u, -v)) % 360


def circular_mean_deg(angles_deg):
    angles_rad = np.radians(angles_deg)
    mean_rad = np.arctan2(
        np.mean(np.sin(angles_rad)),
        np.mean(np.cos(angles_rad))
    )
    return np.degrees(mean_rad) % 360


def circular_resultant_length_deg(angles_deg):
    angles_rad = np.radians(angles_deg)
    return np.sqrt(
        np.mean(np.cos(angles_rad))**2
        + np.mean(np.sin(angles_rad))**2
    )


def circular_sd_deg(angles_deg):
    R = circular_resultant_length_deg(angles_deg)
    R = np.clip(R, 1e-12, 1)
    return np.degrees(np.sqrt(-2 * np.log(R)))


def strip_annotation_suffix(filename):
    name = Path(filename).name.strip()

    if name.endswith(".csv"):
        name = name[:-4]

    if name.endswith("_waggle_annotations"):
        name = name.replace("_waggle_annotations", "")

    return name


def resolve_file(file_value, search_roots, label="file"):
    """
    Resolve a file using priority search roots.

    Allows the mapping file to contain:
    - full path
    - relative path from one of the search roots
    - filename with .csv
    - filename without .csv

    Important:
    Searches roots in order and returns the first unambiguous match.
    This avoids false duplicate errors when the same file exists in several folders.
    """
    if pd.isna(file_value):
        raise ValueError(f"Missing {label} in mapping file.")

    value = str(file_value).strip().replace('"', "").replace("'", "")

    if value == "" or value.lower() == "nan":
        raise ValueError(f"Empty {label} in mapping file.")

    value_path = Path(value)

    # 1. Full path
    if value_path.exists():
        return value_path

    possible_values = [value]

    if not value.endswith(".csv"):
        possible_values.append(value + ".csv")

    # 2. Try exact relative path / direct filename inside each root, in priority order
    for root in search_roots:
        root = Path(root)

        for possible_value in possible_values:
            candidate = root / possible_value

            if candidate.exists():
                return candidate

    # 3. Recursive search, but root by root.
    # Return if one root gives exactly one match.
    all_matches = []

    for root in search_roots:
        root = Path(root)
        root_matches = []

        for possible_value in possible_values:
            possible_name = Path(possible_value).name
            root_matches.extend(root.rglob(possible_name))

        root_matches = sorted(set(root_matches))
        all_matches.extend(root_matches)

        if len(root_matches) == 1:
            return root_matches[0]

        if len(root_matches) > 1:
            raise ValueError(
                f"Found multiple matches for {label}: {value}\n"
                f"Within priority folder:\n{root}\n\n"
                "Use a relative path in the mapping file to disambiguate, for example:\n"
                r"20250121\22b_antisolar_2025-01-21 11-47-07_waggle_annotations\filename.csv"
                "\n\nMatches found:\n"
                + "\n".join(str(match) for match in root_matches)
            )

    raise FileNotFoundError(
        f"Could not find {label}: {value}\n"
        f"Searched in:\n"
        + "\n".join(str(root) for root in search_roots)
    )

In [5]:
def extract_direction_angles_from_csv(csv_file):
    """
    For compass or landmark annotation files.
    Returns one row per annotation arrow.
    """
    csv_file = Path(csv_file)
    df = pd.read_csv(csv_file)

    records = []

    for session_index, row in df.iterrows():
        positions = parse_annotation_list(
            row["waggle_start_positions"],
            "waggle_start_positions"
        )

        frames = parse_annotation_list(
            row["waggle_start_frames"],
            "waggle_start_frames"
        )

        directions = parse_annotation_list(
            row["waggle_directions"],
            "waggle_directions"
        )

        if not (len(positions) == len(frames) == len(directions)):
            raise ValueError(
                f"{csv_file.name}: positions, frames, and directions have different lengths."
            )

        for i, ((x, y), frame, direction) in enumerate(
            zip(positions, frames, directions),
            start=1
        ):
            u, v = direction

            if u is None or v is None:
                continue

            u = float(u)
            v = float(v)

            if not np.isfinite(u) or not np.isfinite(v):
                continue

            if np.hypot(u, v) < 1e-8:
                continue

            video_angle_deg = vector_to_video_angle_deg(u, v)

            records.append({
                "source_file": csv_file.name,
                "annotation_number": len(records) + 1,
                "original_annotation_index": i,
                "frame": frame,
                "x": x,
                "y": y,
                "direction_u": u,
                "direction_v": v,
                "video_angle_deg": video_angle_deg,
                "annotation_session": session_index + 1
            })

    return pd.DataFrame(records)


def extract_waggle_runs(csv_file, source_batch):
    """
    For the actual dome waggle dance file.
    Returns one row per waggle run.
    """
    csv_file = Path(csv_file)
    df = pd.read_csv(csv_file)

    records = []

    dance_id = strip_annotation_suffix(csv_file.name)
    dance_key = f"{source_batch}__{dance_id}"

    waggle_run_number = 0

    for session_index, row in df.iterrows():
        start_positions = parse_annotation_list(
            row["waggle_start_positions"],
            "waggle_start_positions"
        )

        start_frames = parse_annotation_list(
            row["waggle_start_frames"],
            "waggle_start_frames"
        )

        directions = parse_annotation_list(
            row["waggle_directions"],
            "waggle_directions"
        )

        if not (len(start_positions) == len(start_frames) == len(directions)):
            raise ValueError(
                f"{csv_file.name}: start positions, frames, and directions have different lengths."
            )

        for (start_x, start_y), start_frame, direction in zip(
            start_positions,
            start_frames,
            directions
        ):
            u, v = direction

            if u is None or v is None:
                continue

            u = float(u)
            v = float(v)

            if not np.isfinite(u) or not np.isfinite(v):
                continue

            if np.hypot(u, v) < 1e-8:
                continue

            waggle_run_number += 1

            video_angle_deg = vector_to_video_angle_deg(u, v)

            records.append({
                "source_batch": source_batch,
                "dance_key": dance_key,
                "dance_id": dance_id,
                "waggle_run": waggle_run_number,
                "csv_file": csv_file.name,
                "video_name": row["video_name"],
                "direction_u": u,
                "direction_v": v,
                "video_angle_deg": video_angle_deg,
                "start_x": start_x,
                "start_y": start_y,
                "start_xy": (start_x, start_y),
                "start_frame": start_frame,
                "annotation_session": session_index + 1
            })

    return pd.DataFrame(records)

In [6]:
# ---------------------------------------------------------------
# Robust file resolver for landmark-transfer section
# ---------------------------------------------------------------

def resolve_file(file_value, search_roots, label="file"):
    """
    Resolve a FILE using priority search roots.

    Allows mapping file entries as:
    - full path
    - relative path from one of the search roots
    - filename with .csv
    - filename with .CSV
    - filename without extension

    Important fix:
    Candidate paths must be FILES, not just existing paths.
    This prevents the resolver from accidentally returning a folder such as
    need_calc/47y_00073_waggle_annotations when the desired file is inside it.
    """

    if pd.isna(file_value):
        raise ValueError(f"Missing {label} in mapping file.")

    value = str(file_value).strip().replace('"', "").replace("'", "")

    if value == "" or value.lower() == "nan":
        raise ValueError(f"Empty {label} in mapping file.")

    value_path = Path(value)

    # 1. Full path, but only accept actual files
    if value_path.is_file():
        return value_path

    # Build possible filename versions
    possible_values = [value]

    if value.lower().endswith(".csv"):
        stem = value[:-4]
        possible_values.append(stem)
        possible_values.append(stem + ".csv")
        possible_values.append(stem + ".CSV")
    else:
        possible_values.append(value + ".csv")
        possible_values.append(value + ".CSV")

    possible_values = list(dict.fromkeys(possible_values))

    # 2. Direct check in each root, but only accept actual files
    for root in search_roots:
        root = Path(root)

        for possible_value in possible_values:
            candidate = root / possible_value

            if candidate.is_file():
                return candidate

    # 3. Recursive, case-insensitive search in each root
    for root in search_roots:
        root = Path(root)
        root_matches = []

        for file in root.rglob("*"):
            if not file.is_file():
                continue

            file_name_lower = file.name.lower()

            for possible_value in possible_values:
                possible_name_lower = Path(possible_value).name.lower()

                if file_name_lower == possible_name_lower:
                    root_matches.append(file)

        root_matches = sorted(set(root_matches))

        if len(root_matches) == 1:
            return root_matches[0]

        if len(root_matches) > 1:
            raise ValueError(
                f"Found multiple matches for {label}: {value}\n"
                f"Within priority folder:\n{root}\n\n"
                "Use a relative path in the mapping file to disambiguate.\n\n"
                "Matches found:\n"
                + "\n".join(str(match) for match in root_matches)
            )

    raise FileNotFoundError(
        f"Could not find {label}: {value}\n\n"
        "Tried these filename variants:\n"
        + "\n".join(possible_values)
        + "\n\nSearched in:\n"
        + "\n".join(str(root) for root in search_roots)
    )


In [7]:
# ---------------------------------------------------------------
# Landmark-transfer row processor
# ---------------------------------------------------------------

def process_landmark_transfer_row(row):
    orientation_block = str(row["orientation_block"]).strip()

    # 1. Actual waggle-angle annotation file.
    # The resolver must return the CSV file, not the similarly named folder.
    waggle_file = resolve_file(
        row["waggle_file"],
        search_roots=[NEED_CALC_ROOT],
        label="waggle_file"
    )

    waggle_folder = waggle_file.parent

    # 2. Landmark annotated in the actual waggle dance video.
    landmark_dance_file = resolve_file(
        row["landmark_dance_file"],
        search_roots=[waggle_folder],
        label="landmark_dance_file"
    )

    # 3. Landmark annotated in the reference calibration video.
    # Search the row-specific folder first to avoid duplicate matches across dances.
    reference_landmark_file = resolve_file(
        row["reference_landmark_file"],
        search_roots=[waggle_folder],
        label="reference_landmark_file"
    )

    # 4. Compass/north annotation in the reference calibration video.
    # Prefer the copy in the same row-specific folder, but allow COMPASS_ROOT as fallback.
    reference_compass_file = resolve_file(
        row["reference_compass_file"],
        search_roots=[waggle_folder, COMPASS_ROOT],
        label="reference_compass_file"
    )

    # ---------------------------------------------------------------
    # Compass north from reference calibration video
    # ---------------------------------------------------------------

    compass_df = extract_direction_angles_from_csv(reference_compass_file)

    if len(compass_df) == 0:
        raise ValueError(
            f"No valid compass annotations found in {reference_compass_file.name}"
        )

    magnetic_north_calibration_deg = circular_mean_deg(
        compass_df["video_angle_deg"].values
    )

    true_north_calibration_deg = (
        magnetic_north_calibration_deg - DECLINATION_DEG
    ) % 360

    compass_circular_sd = circular_sd_deg(
        compass_df["video_angle_deg"].values
    )

    # ---------------------------------------------------------------
    # Landmark in reference calibration video
    # ---------------------------------------------------------------

    reference_landmark_df = extract_direction_angles_from_csv(
        reference_landmark_file
    )

    if len(reference_landmark_df) == 0:
        raise ValueError(
            f"No valid landmark annotations found in {reference_landmark_file.name}"
        )

    landmark_calibration_deg = circular_mean_deg(
        reference_landmark_df["video_angle_deg"].values
    )

    landmark_calibration_circular_sd = circular_sd_deg(
        reference_landmark_df["video_angle_deg"].values
    )

    north_relative_to_landmark_deg = (
        true_north_calibration_deg - landmark_calibration_deg
    ) % 360

    # ---------------------------------------------------------------
    # Landmark in actual dance video
    # ---------------------------------------------------------------

    dance_landmark_df = extract_direction_angles_from_csv(
        landmark_dance_file
    )

    if len(dance_landmark_df) == 0:
        raise ValueError(
            f"No valid landmark annotations found in {landmark_dance_file.name}"
        )

    landmark_dance_video_deg = circular_mean_deg(
        dance_landmark_df["video_angle_deg"].values
    )

    landmark_dance_circular_sd = circular_sd_deg(
        dance_landmark_df["video_angle_deg"].values
    )

    true_north_video_deg = (
        landmark_dance_video_deg + north_relative_to_landmark_deg
    ) % 360

    # ---------------------------------------------------------------
    # Actual waggle runs
    # ---------------------------------------------------------------

    waggle_df = extract_waggle_runs(
        waggle_file,
        source_batch=orientation_block
    )

    waggle_df["orientation_block"] = orientation_block
    waggle_df["bee_id"] = row.get("bee_id", "")
    waggle_df["condition"] = row.get("condition", "")
    waggle_df["orientation_notes"] = row.get("notes", "")
    waggle_df["orientation_method"] = "landmark_transfer"

    waggle_df["reference_compass_file"] = reference_compass_file.name
    waggle_df["reference_landmark_file"] = reference_landmark_file.name
    waggle_df["landmark_dance_file"] = landmark_dance_file.name

    waggle_df["magnetic_north_calibration_deg"] = magnetic_north_calibration_deg
    waggle_df["true_north_calibration_deg"] = true_north_calibration_deg
    waggle_df["compass_circular_sd_deg"] = compass_circular_sd

    waggle_df["landmark_calibration_deg"] = landmark_calibration_deg
    waggle_df["landmark_calibration_circular_sd_deg"] = landmark_calibration_circular_sd
    waggle_df["north_relative_to_landmark_deg"] = north_relative_to_landmark_deg

    waggle_df["landmark_dance_video_deg"] = landmark_dance_video_deg
    waggle_df["landmark_dance_circular_sd_deg"] = landmark_dance_circular_sd

    waggle_df["true_north_video_deg"] = true_north_video_deg

    waggle_df["waggle_true_bearing_deg"] = (
        waggle_df["video_angle_deg"] - waggle_df["true_north_video_deg"]
    ) % 360

    orientation_summary = {
        "orientation_block": orientation_block,
        "bee_id": row.get("bee_id", ""),
        "condition": row.get("condition", ""),
        "notes": row.get("notes", ""),

        "waggle_file": waggle_file.name,
        "landmark_dance_file": landmark_dance_file.name,
        "reference_landmark_file": reference_landmark_file.name,
        "reference_compass_file": reference_compass_file.name,

        "n_compass_annotations": len(compass_df),
        "magnetic_north_calibration_deg": magnetic_north_calibration_deg,
        "true_north_calibration_deg": true_north_calibration_deg,
        "compass_circular_sd_deg": compass_circular_sd,

        "n_reference_landmark_annotations": len(reference_landmark_df),
        "landmark_calibration_deg": landmark_calibration_deg,
        "landmark_calibration_circular_sd_deg": landmark_calibration_circular_sd,

        "north_relative_to_landmark_deg": north_relative_to_landmark_deg,

        "n_dance_landmark_annotations": len(dance_landmark_df),
        "landmark_dance_video_deg": landmark_dance_video_deg,
        "landmark_dance_circular_sd_deg": landmark_dance_circular_sd,

        "true_north_video_deg": true_north_video_deg,
        "n_waggle_runs": len(waggle_df),
        "orientation_method": "landmark_transfer"
    }

    return waggle_df, orientation_summary


In [8]:
# ---------------------------------------------------------------
# Optional sanity check: resolve landmark-transfer files
# ---------------------------------------------------------------

mapping_test_df = pd.read_csv(
    LANDMARK_MAPPING_FILE,
    dtype=str,
    encoding="utf-8-sig"
)
mapping_test_df.columns = mapping_test_df.columns.str.strip()

for idx, row in mapping_test_df.iterrows():
    print("\n" + "=" * 100)
    print(f"Row {idx + 1}")
    print("bee_id:", row.get("bee_id", ""))
    print("waggle_file from mapping:", row["waggle_file"])

    waggle_file = resolve_file(
        row["waggle_file"],
        search_roots=[NEED_CALC_ROOT],
        label="waggle_file"
    )
    waggle_folder = waggle_file.parent

    landmark_dance_file = resolve_file(
        row["landmark_dance_file"],
        search_roots=[waggle_folder],
        label="landmark_dance_file"
    )

    reference_landmark_file = resolve_file(
        row["reference_landmark_file"],
        search_roots=[waggle_folder],
        label="reference_landmark_file"
    )

    reference_compass_file = resolve_file(
        row["reference_compass_file"],
        search_roots=[waggle_folder, COMPASS_ROOT],
        label="reference_compass_file"
    )

    print("  waggle_folder:            ", waggle_folder)
    print("  waggle_file:              ", waggle_file.name, "| is_file:", waggle_file.is_file())
    print("  landmark_dance_file:      ", landmark_dance_file.name, "| is_file:", landmark_dance_file.is_file())
    print("  reference_landmark_file:  ", reference_landmark_file.name, "| is_file:", reference_landmark_file.is_file())
    print("  reference_compass_file:   ", reference_compass_file.name, "| is_file:", reference_compass_file.is_file())



Row 1
bee_id: 47y
waggle_file from mapping: 47y_00073_waggle_annotations.csv
  waggle_folder:             C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\north_calibrations\horizontal_north_calibrations\horizontal_landmark_transfer_calibrations\47y_00073_waggle_annotations
  waggle_file:               47y_00073_waggle_annotations.csv | is_file: True
  landmark_dance_file:       25_landmark_00073_waggle_annotations.csv | is_file: True
  reference_landmark_file:   25_landmark_calibration_00068_waggle_annotations.csv | is_file: True
  reference_compass_file:    25_00068_waggle_annotations.csv | is_file: True

Row 2
bee_id: 47y
waggle_file from mapping: 47y_00074_waggle_annotations.csv
  waggle_folder:             C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\north_calibrations\horizontal_north_calibrations\horizontal_landmark_transfer_calibrations\47y_00074_waggle_annotations
  waggle_file:               47y_0007

In [9]:
mapping_df = pd.read_csv(LANDMARK_MAPPING_FILE, dtype=str, encoding="utf-8-sig")
mapping_df.columns = mapping_df.columns.str.strip()

required_columns = [
    "orientation_block",
    "waggle_file",
    "landmark_dance_file",
    "reference_landmark_file",
    "bee_id",
    "reference_compass_file"
]

missing_columns = [
    col for col in required_columns
    if col not in mapping_df.columns
]

if missing_columns:
    raise ValueError(
        "Your mapping file is missing these columns:\n"
        + "\n".join(missing_columns)
    )

# Remove completely empty rows
mapping_df = mapping_df.replace(r"^\s*$", pd.NA, regex=True)
mapping_df = mapping_df.dropna(
    subset=required_columns,
    how="all"
).reset_index(drop=True)

all_waggle_dfs = []
orientation_rows = []
failed_rows = []

for idx, row in mapping_df.iterrows():
    try:
        waggle_df, orientation_summary = process_landmark_transfer_row(row)

        all_waggle_dfs.append(waggle_df)
        orientation_rows.append(orientation_summary)

        print(f"Processed row {idx + 1}: {orientation_summary['waggle_file']}")

    except Exception as error:
        print(f"\nCould not process row {idx + 1}")
        print(error)

        failed_rows.append({
            "row_number": idx + 1,
            "error": str(error),
            **row.to_dict()
        })

if len(all_waggle_dfs) == 0:
    raise ValueError("No landmark-transfer videos were processed successfully.")

horizontal_landmark_waggle_df = pd.concat(
    all_waggle_dfs,
    ignore_index=True
)

horizontal_landmark_calibration_summary_df = pd.DataFrame(
    orientation_rows
)

horizontal_landmark_waggle_df.to_csv(
    OUTPUT_WAGGLE_SUMMARY,
    index=False
)

horizontal_landmark_calibration_summary_df.to_csv(
    OUTPUT_ORIENTATION_SUMMARY,
    index=False
)

print(f"\nSaved calibrated landmark-transfer waggle runs to:\n{OUTPUT_WAGGLE_SUMMARY}")
print(f"\nSaved landmark-transfer orientation summary to:\n{OUTPUT_ORIENTATION_SUMMARY}")

if failed_rows:
    failed_df = pd.DataFrame(failed_rows)

    FAILED_ROWS_FILE = OUTPUT_ORIENTATION_SUMMARY.with_name(
        "horizontal_landmark_failed_rows.csv"
    )

    failed_df.to_csv(FAILED_ROWS_FILE, index=False)

    print(f"\nSome rows failed. Saved failed row report to:\n{FAILED_ROWS_FILE}")

Processed row 1: 47y_00073_waggle_annotations.csv
Processed row 2: 47y_00074_waggle_annotations.csv
Processed row 3: 11r_00010_waggle_annotations.csv
Processed row 4: 25y_00020_waggle_annotations.csv

Saved calibrated landmark-transfer waggle runs to:
C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\horizontal_landmark_calibrated_waggle_runs.csv

Saved landmark-transfer orientation summary to:
C:\Users\frida\Documents\PhD\Waggle_dance\wd_SA2025\wd_annotator\CORRECTED_csv_files\horizontal_landmark_calibration_summary.csv


### Scratch folder listing removed
This cell intentionally replaces an old manual folder listing.
